# External evaluation on DDI

Scores every trained model on the 656 images of the Diverse Dermatology Images dataset and repeats the analysis of notebook 00. No DDI image is used for training or model selection. The last part is the sensitivity analysis without the non-neoplastic DDI images.

The prediction step needs the checkpoints and PyTorch. Once `ddi_predictions.csv` exists in every run folder, the rest runs without a GPU.

In [1]:
import sys
from pathlib import Path

# The shared code is in the repository root. Look for it from the folder of this notebook upwards.
REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "analysis.py").is_file())
sys.path.insert(0, str(REPO_DIR))

import pandas as pd
import analysis as A
import ddi as D

SUMMARY_DIR = A.SUMMARY_DIR
SUMMARY_DIR.mkdir(exist_ok=True)
pd.set_option("display.float_format", "{:.4f}".format)

ddi_df = D.load_metadata()
description = D.describe(ddi_df)
description.to_csv(SUMMARY_DIR / "ddi_description.csv")
description

,n,n_malignant,n_non_neoplastic,n_after_exclusion
fst_group,,,,
FST 1-2,208,49,30,178
FST 3-4,241,74,17,224
FST 5-6,207,48,42,165
All,656,171,89,567


In [2]:
# Diagnoses that are left out in the sensitivity analysis, with the number of images per FST group.
ddi_df[ddi_df.non_neoplastic].groupby(["disease", "fst_group"]).size().unstack(fill_value=0)

fst_group,FST 1-2,FST 3-4,FST 5-6
disease,,,
abrasions-ulcerations-and-physical-injuries,0,0,3
abscess,0,1,0
acne-cystic,0,0,1
coccidioidomycosis,0,1,0
condyloma-accuminatum,1,0,1
dermatomyositis,0,1,0
eczema-spongiotic-dermatitis,0,0,4
focal-acral-hyperkeratosis,0,0,1
folliculitis,0,0,1


## Predictions

In [3]:
# About a minute per run on a GPU. Runs that already have ddi_predictions.csv are skipped.
runs = D.predict_all(ddi_df)

baseline                         seed 42: ddi_predictions.csv
baseline                         seed 43: ddi_predictions.csv
baseline                         seed 44: ddi_predictions.csv
universal_colour                 seed 42: ddi_predictions.csv
universal_colour                 seed 43: ddi_predictions.csv
universal_colour                 seed 44: ddi_predictions.csv
targeted_colour                  seed 42: ddi_predictions.csv
targeted_colour                  seed 43: ddi_predictions.csv
targeted_colour                  seed 44: ddi_predictions.csv
reweighting_effective_number     seed 42: ddi_predictions.csv
reweighting_effective_number     seed 43: ddi_predictions.csv
reweighting_effective_number     seed 44: ddi_predictions.csv
reweighting_kamiran              seed 42: ddi_predictions.csv
reweighting_kamiran              seed 43: ddi_predictions.csv
reweighting_kamiran              seed 44: ddi_predictions.csv
combined                         seed 42: ddi_predictions.csv
combined

In [4]:
per_run = D.per_run_table(runs)
per_run.to_csv(SUMMARY_DIR / "ddi_per_run_results.csv", index=False)
per_run

,condition,seed,n,auc,auc_fst_1_2,auc_fst_3_4,auc_fst_5_6,auc_gap_max_min,auc_gap_1_2_minus_5_6
0,baseline,42,656,0.6674,0.6360,0.7017,0.6490,0.0657,-0.0130
1,baseline,43,656,0.6386,0.6021,0.6694,0.6360,0.0673,-0.0339
2,baseline,44,656,0.6499,0.6583,0.6597,0.6257,0.0340,0.0327
3,universal_colour,42,656,0.6778,0.6124,0.7011,0.7076,0.0952,-0.0952
4,universal_colour,43,656,0.6690,0.6002,0.7062,0.6850,0.1060,-0.0848
5,universal_colour,44,656,0.6581,0.6015,0.6795,0.6791,0.0780,-0.0777
6,targeted_colour,42,656,0.6714,0.6430,0.7114,0.6437,0.0685,-0.0008
7,targeted_colour,43,656,0.6704,0.6420,0.7199,0.6296,0.0904,0.0124
8,targeted_colour,44,656,0.6367,0.6172,0.6674,0.6166,0.0508,0.0006
9,reweighting_effective_number,42,656,0.6544,0.6549,0.6847,0.6107,0.0740,0.0441


## Main analysis: all 656 images

In [5]:
est_all, diff_all = D.bootstrap(runs, exclude_non_neoplastic=False)
est_all.to_csv(SUMMARY_DIR / "ddi_bootstrap_auc.csv", index=False)
diff_all.to_csv(SUMMARY_DIR / "ddi_bootstrap_diff_vs_baseline.csv", index=False)
A.format_ci_table(est_all)

,condition,auc,auc_fst_1_2,auc_fst_3_4,auc_fst_5_6,auc_gap_1_2_minus_5_6
0,baseline,"0.652 [0.607, 0.699]","0.632 [0.545, 0.721]","0.677 [0.608, 0.746]","0.637 [0.548, 0.723]","-0.005 [-0.127, 0.121]"
1,universal_colour,"0.668 [0.625, 0.712]","0.605 [0.521, 0.693]","0.696 [0.628, 0.762]","0.691 [0.614, 0.770]","-0.086 [-0.201, 0.028]"
2,targeted_colour,"0.659 [0.615, 0.704]","0.634 [0.549, 0.720]","0.700 [0.632, 0.766]","0.630 [0.544, 0.716]","0.004 [-0.122, 0.127]"
3,reweighting_effective_number,"0.645 [0.599, 0.691]","0.638 [0.554, 0.724]","0.675 [0.603, 0.745]","0.612 [0.529, 0.695]","0.026 [-0.092, 0.142]"
4,reweighting_kamiran,"0.653 [0.607, 0.697]","0.643 [0.560, 0.729]","0.676 [0.604, 0.746]","0.627 [0.542, 0.712]","0.016 [-0.102, 0.138]"
5,combined,"0.655 [0.608, 0.700]","0.610 [0.526, 0.699]","0.696 [0.624, 0.764]","0.644 [0.559, 0.732]","-0.034 [-0.158, 0.088]"


In [6]:
# Paired differences from the baseline. A star means that the 95% interval excludes zero.
A.format_diff_table(diff_all)

,condition,auc,auc_fst_1_2,auc_fst_3_4,auc_fst_5_6,auc_gap_1_2_minus_5_6
0,universal_colour,"+0.016 [-0.002, +0.035]","-0.027 [-0.060, +0.002]","+0.019 [-0.008, +0.047]","+0.054 [+0.014, +0.095]*","-0.081 [-0.134, -0.032]*"
1,targeted_colour,"+0.008 [-0.009, +0.023]","+0.002 [-0.029, +0.033]","+0.023 [+0.001, +0.046]*","-0.007 [-0.042, +0.026]","+0.009 [-0.037, +0.056]"
2,reweighting_effective_number,"-0.007 [-0.023, +0.009]","+0.006 [-0.018, +0.028]","-0.002 [-0.027, +0.022]","-0.025 [-0.058, +0.011]","+0.030 [-0.013, +0.071]"
3,reweighting_kamiran,"+0.001 [-0.012, +0.013]","+0.011 [-0.013, +0.035]","-0.001 [-0.018, +0.016]","-0.010 [-0.034, +0.016]","+0.021 [-0.015, +0.055]"
4,combined,"+0.003 [-0.018, +0.023]","-0.022 [-0.058, +0.012]","+0.019 [-0.013, +0.049]","+0.007 [-0.033, +0.051]","-0.029 [-0.086, +0.024]"


## Sensitivity analysis: without the non-neoplastic images

In [7]:
est_sens, diff_sens = D.bootstrap(runs, exclude_non_neoplastic=True)
est_sens.to_csv(SUMMARY_DIR / "ddi_sensitivity_bootstrap_auc.csv", index=False)
diff_sens.to_csv(SUMMARY_DIR / "ddi_sensitivity_bootstrap_diff_vs_baseline.csv", index=False)
A.format_ci_table(est_sens)

,condition,auc,auc_fst_1_2,auc_fst_3_4,auc_fst_5_6,auc_gap_1_2_minus_5_6
0,baseline,"0.655 [0.606, 0.703]","0.637 [0.544, 0.729]","0.683 [0.607, 0.752]","0.635 [0.537, 0.725]","0.003 [-0.130, 0.134]"
1,universal_colour,"0.670 [0.623, 0.716]","0.614 [0.519, 0.704]","0.698 [0.626, 0.767]","0.683 [0.595, 0.764]","-0.070 [-0.195, 0.057]"
2,targeted_colour,"0.662 [0.613, 0.709]","0.640 [0.551, 0.728]","0.702 [0.630, 0.769]","0.629 [0.534, 0.721]","0.011 [-0.119, 0.144]"
3,reweighting_effective_number,"0.646 [0.597, 0.694]","0.644 [0.554, 0.733]","0.677 [0.603, 0.750]","0.606 [0.513, 0.694]","0.038 [-0.087, 0.164]"
4,reweighting_kamiran,"0.654 [0.605, 0.701]","0.648 [0.556, 0.738]","0.679 [0.604, 0.750]","0.625 [0.532, 0.713]","0.023 [-0.104, 0.151]"
5,combined,"0.660 [0.610, 0.708]","0.623 [0.533, 0.715]","0.701 [0.628, 0.773]","0.641 [0.542, 0.727]","-0.018 [-0.149, 0.113]"


In [8]:
A.format_diff_table(diff_sens)

,condition,auc,auc_fst_1_2,auc_fst_3_4,auc_fst_5_6,auc_gap_1_2_minus_5_6
0,universal_colour,"+0.016 [-0.003, +0.034]","-0.024 [-0.055, +0.007]","+0.016 [-0.012, +0.044]","+0.049 [+0.009, +0.089]*","-0.073 [-0.123, -0.022]*"
1,targeted_colour,"+0.007 [-0.009, +0.024]","+0.002 [-0.029, +0.037]","+0.019 [-0.004, +0.042]","-0.006 [-0.041, +0.031]","+0.008 [-0.039, +0.057]"
2,reweighting_effective_number,"-0.008 [-0.024, +0.008]","+0.006 [-0.018, +0.030]","-0.005 [-0.030, +0.018]","-0.029 [-0.062, +0.007]","+0.035 [-0.009, +0.076]"
3,reweighting_kamiran,"-0.001 [-0.012, +0.011]","+0.011 [-0.016, +0.035]","-0.004 [-0.022, +0.013]","-0.010 [-0.035, +0.016]","+0.021 [-0.017, +0.058]"
4,combined,"+0.006 [-0.014, +0.026]","-0.015 [-0.049, +0.021]","+0.018 [-0.015, +0.050]","+0.006 [-0.037, +0.049]","-0.021 [-0.075, +0.036]"


In [9]:
# Both analyses side by side: does leaving out the non-neoplastic images change the estimates?
cols = A.REPORTED_STATS
compare = est_all.set_index("condition")[cols].add_suffix("_all").join(
    est_sens.set_index("condition")[cols].add_suffix("_excl"))
compare = compare[[c + s for c in cols for s in ("_all", "_excl")]]
compare.to_csv(SUMMARY_DIR / "ddi_sensitivity_comparison.csv")
compare

,auc_all,auc_excl,auc_fst_1_2_all,auc_fst_1_2_excl,auc_fst_3_4_all,auc_fst_3_4_excl,auc_fst_5_6_all,auc_fst_5_6_excl,auc_gap_1_2_minus_5_6_all,auc_gap_1_2_minus_5_6_excl
condition,,,,,,,,,,
baseline,0.6520,0.6548,0.6321,0.6373,0.6769,0.6829,0.6369,0.6346,-0.0047,0.0027
universal_colour,0.6683,0.6703,0.6047,0.6136,0.6956,0.6985,0.6906,0.6834,-0.0859,-0.0698
targeted_colour,0.6595,0.6618,0.6341,0.6397,0.6996,0.7023,0.6300,0.6289,0.0041,0.0107
reweighting_effective_number,0.6448,0.6465,0.6380,0.6438,0.6750,0.6774,0.6122,0.6061,0.0258,0.0378
reweighting_kamiran,0.6526,0.6539,0.6429,0.6480,0.6764,0.6786,0.6267,0.6246,0.0162,0.0234
combined,0.6549,0.6603,0.6103,0.6227,0.6959,0.7013,0.6439,0.6407,-0.0337,-0.0180
